# Kaggle Runner for BCC Skin Lesion Detection Dashboard

This notebook guides you through setting up, training, and running the BCC Skin Lesion Detection interactive web dashboard directly on **Kaggle** utilizing a free GPU accelerator.

### Kaggle Settings Checklist before running:
1. **Accelerator**: Select **GPU T4 x2** or **GPU P100** on the right settings panel.
2. **Internet on**: Ensure the toggle is switched **ON** (required to install packages, download pre-trained weights, and establish the web tunnel).

### How to use this notebook:
1. Click **File** -> **Upload data** and upload the compact **`BCCPROJECT_code_only.zip`** file.
2. Run the setup cells in order to extract code, install libraries, and prepare directories.
3. Run training and evaluation (optional, requires uploading your dataset).
4. Start the interactive dashboard and click the generated Pinggy URL to access the web portal.

## Step 1: Unzip the Project Code
Run this cell to copy the uploaded zip file from the input folder to the writable `/kaggle/working` directory and extract it.

In [ ]:
import os
import zipfile

# Locate the uploaded code zip file
zip_name = 'BCCPROJECT_code_only.zip'
zip_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if zip_name in files:
        zip_path = os.path.join(root, zip_name)
        break

extract_dir = '/kaggle/working/BCCPROJECT'

if zip_path:
    print(f"Found zip at {zip_path}. Extracting to {extract_dir}...")
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Extraction complete!")
else:
    print(f"WARNING: '{zip_name}' was not found in '/kaggle/input'.")
    print("If you cloned the repo or uploaded the code directly, make sure you navigate to the proper directory.")
    # Fallback to current working directory if already extracted
    if os.path.exists('/kaggle/working/BCCPROJECT'):
        print("Using existing directory: /kaggle/working/BCCPROJECT")
    else:
        print("Please upload 'BCCPROJECT_code_only.zip' as a dataset first.")

# Navigate into the project directory
%cd /kaggle/working/BCCPROJECT

## Step 2: Install Required Dependencies

In [ ]:
!pip install timm albumentations flask fastapi uvicorn

## Step 3: Run Offline Preprocessing & Model Training (Optional)
If you uploaded your dataset to Kaggle under `/kaggle/input/` or `/kaggle/working/BCCPROJECT/dataset`, you can preprocess and train the model here.

In [ ]:
# Preprocess files offline to speed up training I/O by 10x
# !python scripts/preprocess_offline.py

# Train the lightweight model on the GPU
# !python -u scripts/train_fast.py

## Step 4: Run Evaluation (Optional)
Run tests on the validation/test partitions and generate report metrics.

In [ ]:
# !python scripts/evaluate_test_set.py

## Step 5: Launch the Diagnostic Web Dashboard (with Public URL Tunnel)
Kaggle doesn't provide automatic web proxy routing. To access the dashboard, we create a secure SSH tunnel to port 5000 using **Pinggy** (free, no account setup required).

Run the cell below, wait a few seconds, and click the generated **`https://...`** URL to open the dashboard.

In [ ]:
import subprocess
import time

# 1. Start the Flask application in the background
print("Starting Flask application server on port 5000...")
flask_process = subprocess.Popen(["python", "app.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(4)  # Give the server time to load the model checkpoint

# 2. Establish a tunnel with Pinggy
print("Creating public web tunnel via Pinggy...")
tunnel_process = subprocess.Popen(
    ["ssh", "-p", "443", "-o", "StrictHostKeyChecking=no", "-R", "80:localhost:5000", "a.pinggy.io"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# 3. Read stdout line by line until the public URL is printed
for _ in range(30):
    line = tunnel_process.stdout.readline()
    if "http://" in line or "https://" in line:
        print("\n=============================================================")
        print("  BCC SKIN CANCER DASHBOARD TUNNEL CREATED!")
        print(f"  Click here: {line.strip()}")
        print("=============================================================\n")
        break
    time.sleep(0.5)
else:
    print("Could not extract tunnel URL. Please check if SSH outgoing connections are allowed (Internet ON).")

## Step 6: Download Checkpoints and Reports
Run this cell to generate download links for any newly trained model weights, evaluation reports, or curves.

In [ ]:
from IPython.display import FileLink

if os.path.exists('outputs/best_model.pth'):
    display(FileLink('outputs/best_model.pth'))
if os.path.exists('outputs/final_report.md'):
    display(FileLink('outputs/final_report.md'))
if os.path.exists('outputs/roc_curve.png'):
    display(FileLink('outputs/roc_curve.png'))
if os.path.exists('outputs/confusion_matrix.png'):
    display(FileLink('outputs/confusion_matrix.png'))